In [0]:
# --- PASO 1: Instalar la herramienta para bajar datos ---
%pip install yfinance

# --- PASO 2: Importar librerías ---
import yfinance as yf
from pyspark.sql.functions import current_timestamp, lit

# --- PASO 3: Descargar datos de NVIDIA ---
ticker = "NVDA"
raw_data = yf.download(ticker, period="2y", interval="1d")

# CHECK: Verify data was downloaded
if raw_data.empty:
    raise ValueError(
        f"No data retrieved for {ticker}. "
        "Possible causes: API rate limit, network issue, or invalid ticker. "
        "Try again in a few minutes."
    )

print(f"Downloaded {len(raw_data)} rows for {ticker}")

# --- PASO 4: Limpieza de Columnas  ---
df_pd = raw_data.reset_index()
df_pd.columns = [col[0] if isinstance(col, tuple) else col for col in df_pd.columns]

# Ahora sí, convertimos a Spark con nombres limpios
df_spark = spark.createDataFrame(df_pd)

# Añadimos metadatos
df_spark = df_spark.withColumn("ingestion_timestamp", current_timestamp()) \
                   .withColumn("source_name", lit("Yahoo Finance"))

# --- PASO 5: Guardar en la Capa Bronze ---
df_spark.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_nvda_prices")

print("¡Ahora sí! Datos guardados sin errores en: bronze_nvda_prices")

In [0]:
%sql
DESCRIBE TABLE bronze_nvda_prices